In [1]:
import os
import re
import time
import hashlib
from getpass import getpass
from itertools import product

import pandas as pd
from neo4j import GraphDatabase

In [2]:
NEO4J_URI="neo4j://127.0.0.1:7687"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="Partha_1234#"

In [ ]:

NEO4J_URI = os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD") or getpass("Neo4j password: ")

In [3]:
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [28]:
DATA_CSV_SOURCE = "https://raw.githubusercontent.com/SuddhasattwaDas/Lib_GRAPH/refs/heads/main/lisjkg_dataset_cleaned.csv"

In [5]:
RELATIONSHIP_SCHEMA_PATH = "F Relationship.xlsx"

In [6]:
BATCH_SIZE = 3000          # rows per UNWIND transaction
STORE_INVERSE_EDGES = False

In [31]:
def load_bibliographic_data(source):
    # Read directly from URL or local file
    df = pd.read_csv(
        source,
        encoding="utf-8-sig",
        engine="python",
        on_bad_lines="skip"
    )

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(" ", "_", regex=False)
    )

    # Required columns
    required = [
        "Authors","Title","Source","Publisher","Place","Country",
        "ArticleURL","DOI","ISSN","Keywords","Abstract","Year","Cites"
    ]

    for col in required:
        if col not in df.columns:
            df[col] = ""

    # Clean text columns
    text_cols = [
        "Authors","Title","Source","Publisher","Place","Country",
        "ArticleURL","DOI","ISSN","Keywords","Abstract"
    ]

    for col in text_cols:
        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
            .str.replace('"',"",regex=False)
            .str.replace("\r"," ",regex=False)
            .str.replace("\n"," ",regex=False)
            .str.strip()
        )

    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    df["Cites"] = pd.to_numeric(df["Cites"], errors="coerce").fillna(0).astype(int)

    df["Place"] = df.apply(
        lambda r: _combine_place(r["Place"], r["Country"]),
        axis=1
    )

    def make_uid(row):
        if row["DOI"]:
            return row["DOI"]

        if row["Title"]:
            return f"{row['Title']}|{row['Year']}"

        return "UNK_" + hashlib.md5(
            f"{row['Title']}{row['Year']}{row['ArticleURL']}".encode()
        ).hexdigest()[:12]

    df["title_uid"] = df.apply(make_uid, axis=1)

    return df

In [32]:
t0 = time.time()
df = load_bibliographic_data(DATA_CSV_SOURCE)
print(f"Loaded {len(df):,} rows in {time.time() - t0:.1f}s")
print(f"Columns: {list(df.columns)}")
print(f"Unique titles: {df['title_uid'].nunique():,}  |  Year range: {df['Year'].min()}–{df['Year'].max()}")
print(f"Unique combined places: {df['Place'].nunique()}")
df.head(3)

Loaded 27,815 rows in 14.3s
Columns: ['Cites', 'Authors', 'Title', 'Year', 'Source', 'Publisher', 'Place', 'Country', 'ArticleURL', 'DOI', 'ISSN', 'Keywords', 'Abstract', 'title_uid']
Unique titles: 27,815  |  Year range: 2020–2025
Unique combined places: 35


,Cites,Authors,Title,Year,Source,Publisher,Place,Country,ArticleURL,DOI,ISSN,Keywords,Abstract,title_uid
0,18,Patit Paban Santra; Debasis Majhi,Scholarly Communication and Machine-Generated ...,2023,Journal of Information and Knowledge,Sarada Ranganathan Endowment for Library Scien...,"Bangalore, India",India,https://doi.org/10.17821/srels/2023/v60i3/171028,10.17821/srels/2023/v60i3/171028,2583-9314,language model; AI model; AI writing; AI writi...,This study utilizes GPT (Generative Pre-Traine...,10.17821/srels/2023/v60i3/171028
1,8,Bolaji David Oladokun; Joel Eleojo Aruwa; Gabr...,Misinformation and Disinformation in the Era o...,2024,Journal of Information and Knowledge,Sarada Ranganathan Endowment for Library Scien...,"Bangalore, India",India,https://doi.org/10.17821/srels/2024/v61i1/171266,10.17821/srels/2024/v61i1/171266,2583-9314,Fact-Checking Skills; Misinformation; Social M...,The paper explores the need for fact-checking ...,10.17821/srels/2024/v61i1/171266
2,8,Manoj Kumar Verma; Mayank Yuvaraj,AI-Based Literature Reviews: A Topic Modeling ...,2023,Journal of Information and Knowledge,Sarada Ranganathan Endowment for Library Scien...,"Bangalore, India",India,https://doi.org/10.17821/srels/2023/v60i2/170967,10.17821/srels/2023/v60i2/170967,2583-9314,Topic Modelling; Conducting Literature Reviews...,The purpose of this paper is to highlight the ...,10.17821/srels/2023/v60i2/170967


In [15]:
NAME_FIX = {"Publication Year": "Year", "Keywords": "Keyword"}


def load_relationship_schema(path: str):
    """Read F_Relationship.xlsx and collapse it to one canonical edge per (Source, Target) pair."""
    schema_df = pd.read_excel(path)
    schema_df.columns = [c.strip() for c in schema_df.columns]
    for c in ["Source", "Relation", "Target"]:
        schema_df[c] = schema_df[c].astype(str).str.strip().replace(NAME_FIX)

    seen, canonical = {}, []
    for _, r in schema_df.iterrows():
        s, rel, t = r["Source"], r["Relation"], r["Target"]
        pair_key = tuple(sorted([s, t]))
        if pair_key not in seen:
            seen[pair_key] = {"source": s, "relation": rel.upper(), "target": t, "inverse_relation": None}
            canonical.append(seen[pair_key])
        else:
            seen[pair_key]["inverse_relation"] = rel
    return canonical


canonical_schema = load_relationship_schema(RELATIONSHIP_SCHEMA_PATH)
print(f"{len(canonical_schema)} canonical relationship types:\n")
for e in canonical_schema:
    print(f"  ({e['source']})-[:{e['relation']}]->({e['target']})   (inverse label: {e['inverse_relation']})")

29 canonical relationship types:

  (Author)-[:WRITES]->(Title)   (inverse label: is_written_by)
  (Author)-[:CONTRIBUTES_TO]->(Journal)   (inverse label: has_contributor)
  (Author)-[:PUBLISHES_WITH]->(Publisher)   (inverse label: publishes_work_of)
  (Author)-[:AFFILIATED_WITH]->(Place)   (inverse label: hosts_publications_of)
  (Author)-[:USES]->(Keyword)   (inverse label: associated_with)
  (Author)-[:PUBLISHES_IN]->(Year)   (inverse label: has_publication_by)
  (Author)-[:LINKED_BY]->(DOI)   (inverse label: linked_work_of)
  (Author)-[:ACCESSIBLE_VIA]->(URL)   (inverse label: links_to_work_of)
  (Title)-[:PUBLISHED_IN]->(Journal)   (inverse label: contains)
  (Title)-[:PUBLISHED_BY]->(Publisher)   (inverse label: publishes)
  (Title)-[:PUBLISHED_AT]->(Place)   (inverse label: hosts_publication)
  (Title)-[:DESCRIBED_BY]->(Keyword)   (inverse label: describes)
  (Title)-[:APPEARS_IN]->(Year)   (inverse label: includes_title)
  (Title)-[:IDENTIFIED_BY]->(DOI)   (inverse label: repre

In [33]:
NODE_FIELD_MAP = {
    "Author":    {"column": "Authors",    "multi": True,  "delim": ";"},
    "Title":     {"column": "title_uid",  "multi": False, "delim": None},
    "Journal":   {"column": "Source",     "multi": False, "delim": None},
    "Publisher": {"column": "Publisher",  "multi": False, "delim": None},
    "Place":     {"column": "Place",      "multi": False, "delim": None},
    "Keyword":   {"column": "Keywords",   "multi": True,  "delim": ";"},
    "Year":      {"column": "Year",       "multi": False, "delim": None},
    "DOI":       {"column": "DOI",        "multi": False, "delim": None},
    "URL":       {"column": "ArticleURL", "multi": False, "delim": None},
}

LABELS = {entity_type: entity_type for entity_type in NODE_FIELD_MAP}

KEY_PROPS = {
    "Author": "name", "Title": "uid", "Journal": "name", "Publisher": "name",
    "Place": "name", "Keyword": "name", "Year": "value", "DOI": "value", "URL": "value",
}

KEY_CASTERS = {"Year": int}  # Year is stored as an int, so pair-building must cast to int too

_HAS_LETTER = re.compile(r"[A-Za-z]")


def _cast_key(entity_type, value):
    caster = KEY_CASTERS.get(entity_type)
    return caster(value) if caster else value


def extract_all(df: pd.DataFrame) -> dict:
    """For every entity type, return a Series (aligned to df.index) of the list of values found per row."""
    extracted = {}
    for entity_type, spec in NODE_FIELD_MAP.items():
        col = df[spec["column"]]
        if spec["multi"]:
            extracted[entity_type] = col.apply(
                lambda v: [p.strip() for p in str(v).split(spec["delim"]) if p.strip()] if pd.notna(v) else [])
        else:
            extracted[entity_type] = col.apply(
                lambda v: [str(v).strip()] if pd.notna(v) and str(v).strip() and str(v).strip().lower() != "nan" else [])

    # Drop author "names" that contain no letters at all (e.g. "- -", "?. ?. ?????????") --
    # placeholder entries already present in the source CSV, not real names.
    before = sum(len(v) for v in extracted["Author"])
    extracted["Author"] = extracted["Author"].apply(lambda names: [n for n in names if _HAS_LETTER.search(n)])
    dropped = before - sum(len(v) for v in extracted["Author"])
    if dropped:
        print(f"Dropped {dropped} placeholder author mentions with no letters (e.g. '- -')")

    return extracted


def chunked(seq, size):
    seq = list(seq)
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


extracted = extract_all(df)

Dropped 155 placeholder author mentions with no letters (e.g. '- -')


In [34]:
def build_node_payloads(df: pd.DataFrame, extracted: dict) -> dict:
    """One payload list per entity type, ready to UNWIND + MERGE."""
    payloads = {
        "Author": [{"name": a} for a in sorted(set().union(*extracted["Author"]))],
        "Keyword": [{"name": k} for k in sorted(set().union(*extracted["Keyword"]))],
        "Publisher": [{"name": p} for p in sorted({v[0] for v in extracted["Publisher"] if v})],
        "Place": [{"name": p} for p in sorted({v[0] for v in extracted["Place"] if v})],
        "Year": [{"value": int(y)} for y in sorted({v[0] for v in extracted["Year"] if v})],
        "DOI": [{"value": d} for d in sorted({v[0] for v in extracted["DOI"] if v})],
        "URL": [{"value": u} for u in sorted({v[0] for v in extracted["URL"] if v})],
    }

    journal_issn = df[["Source", "ISSN"]].drop_duplicates("Source")
    payloads["Journal"] = [{"name": r["Source"], "issn": r["ISSN"]} for _, r in journal_issn.iterrows() if r["Source"]]

    titles = df[["title_uid", "Title", "Abstract", "Cites"]].drop_duplicates("title_uid")
    payloads["Title"] = [
        {"uid": r["title_uid"], "title": r["Title"], "abstract": r["Abstract"], "cites": int(r["Cites"])}
        for _, r in titles.iterrows()
    ]
    return payloads


def build_relationship_pairs(canonical_schema: list, extracted: dict) -> dict:
    """For every canonical edge, the deduplicated set of (source_value, target_value) pairs across all rows."""
    rel_pairs = {}
    for edge in canonical_schema:
        s_type, rel, t_type = edge["source"], edge["relation"], edge["target"]
        if s_type not in NODE_FIELD_MAP or t_type not in NODE_FIELD_MAP:
            print(f"  [skip] '{s_type}' or '{t_type}' has no NODE_FIELD_MAP entry yet")
            continue
        s_col, t_col = extracted[s_type], extracted[t_type]
        pairs = set()
        for s_list, t_list in zip(s_col, t_col):
            if not s_list or not t_list:
                continue
            for s, t in product(s_list, t_list):
                pairs.add((_cast_key(s_type, s), _cast_key(t_type, t)))
        rel_pairs[(s_type, rel, t_type)] = pairs
    return rel_pairs


node_payloads = build_node_payloads(df, extracted)
relationship_pairs = build_relationship_pairs(canonical_schema, extracted)

print("NODES\n" + "-" * 40)
total_nodes = 0
for label, rows in node_payloads.items():
    print(f"  {label:<10} {len(rows):>8,}")
    total_nodes += len(rows)
print(f"  {'TOTAL':<10} {total_nodes:>8,}")

print("\nRELATIONSHIPS\n" + "-" * 40)
total_rels = 0
for (s, rel, t), pairs in relationship_pairs.items():
    print(f"  ({s})-[:{rel}]->({t}): {len(pairs):,}")
    total_rels += len(pairs)
print(f"  TOTAL: {total_rels:,}")

NODES
----------------------------------------
  Author       52,821
  Keyword      38,589
  Publisher        32
  Place            35
  Year              6
  DOI          27,749
  URL          27,813
  Journal         100
  Title        27,815
  TOTAL       174,960

RELATIONSHIPS
----------------------------------------
  (Author)-[:WRITES]->(Title): 81,028
  (Author)-[:CONTRIBUTES_TO]->(Journal): 66,346
  (Author)-[:PUBLISHES_WITH]->(Publisher): 61,940
  (Author)-[:AFFILIATED_WITH]->(Place): 62,881
  (Author)-[:USES]->(Keyword): 236,412
  (Author)-[:PUBLISHES_IN]->(Year): 68,332
  (Author)-[:LINKED_BY]->(DOI): 80,865
  (Author)-[:ACCESSIBLE_VIA]->(URL): 81,023
  (Title)-[:PUBLISHED_IN]->(Journal): 27,815
  (Title)-[:PUBLISHED_BY]->(Publisher): 27,815
  (Title)-[:PUBLISHED_AT]->(Place): 27,815
  (Title)-[:DESCRIBED_BY]->(Keyword): 83,359
  (Title)-[:APPEARS_IN]->(Year): 27,815
  (Title)-[:IDENTIFIED_BY]->(DOI): 27,749
  (Title)-[:AVAILABLE_AT]->(URL): 27,813
  (Journal)-[:ISSUED_BY]->

In [35]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

try:
    driver.verify_connectivity()
    print(f"Connected to {NEO4J_URI}")
except Exception as exc:
    print(f"Could not connect to Neo4j at {NEO4J_URI}: {exc}")
    raise


def create_constraints(driver):
    with driver.session() as session:
        for entity_type, label in LABELS.items():
            key = KEY_PROPS[entity_type]
            name = f"{label.lower()}_{key}_unique"
            stmt = f"CREATE CONSTRAINT {name} IF NOT EXISTS FOR (n:{label}) REQUIRE n.{key} IS UNIQUE"
            session.run(stmt)
            print(f"  \u2713 {stmt}")


create_constraints(driver)

Connected to neo4j://127.0.0.1:7687
  ✓ CREATE CONSTRAINT author_name_unique IF NOT EXISTS FOR (n:Author) REQUIRE n.name IS UNIQUE
  ✓ CREATE CONSTRAINT title_uid_unique IF NOT EXISTS FOR (n:Title) REQUIRE n.uid IS UNIQUE
  ✓ CREATE CONSTRAINT journal_name_unique IF NOT EXISTS FOR (n:Journal) REQUIRE n.name IS UNIQUE
  ✓ CREATE CONSTRAINT publisher_name_unique IF NOT EXISTS FOR (n:Publisher) REQUIRE n.name IS UNIQUE
  ✓ CREATE CONSTRAINT place_name_unique IF NOT EXISTS FOR (n:Place) REQUIRE n.name IS UNIQUE
  ✓ CREATE CONSTRAINT keyword_name_unique IF NOT EXISTS FOR (n:Keyword) REQUIRE n.name IS UNIQUE
  ✓ CREATE CONSTRAINT year_value_unique IF NOT EXISTS FOR (n:Year) REQUIRE n.value IS UNIQUE
  ✓ CREATE CONSTRAINT doi_value_unique IF NOT EXISTS FOR (n:DOI) REQUIRE n.value IS UNIQUE
  ✓ CREATE CONSTRAINT url_value_unique IF NOT EXISTS FOR (n:URL) REQUIRE n.value IS UNIQUE


In [36]:
def load_nodes(driver, node_payloads: dict, batch_size: int = BATCH_SIZE):
    with driver.session() as session:
        for entity_type, rows in node_payloads.items():
            if not rows:
                print(f"  (skipping {entity_type}: nothing to load)")
                continue
            label = LABELS[entity_type]
            key = KEY_PROPS[entity_type]
            other_props = [p for p in rows[0] if p != key]
            set_clause = ("SET " + ", ".join(f"n.{p} = row.{p}" for p in other_props)) if other_props else ""

            query = f"""
            UNWIND $rows AS row
            MERGE (n:{label} {{{key}: row.{key}}})
            {set_clause}
            """

            t0 = time.time()
            for batch in chunked(rows, batch_size):
                session.run(query, rows=batch)
            print(f"  \u2713 {label:<10} {len(rows):>8,} nodes merged in {time.time() - t0:.1f}s")


load_nodes(driver, node_payloads)

  ✓ Author       52,821 nodes merged in 9.5s
  ✓ Keyword      38,589 nodes merged in 3.2s
  ✓ Publisher        32 nodes merged in 0.4s
  ✓ Place            35 nodes merged in 0.1s
  ✓ Year              6 nodes merged in 0.2s
  ✓ DOI          27,749 nodes merged in 2.0s
  ✓ URL          27,813 nodes merged in 1.7s
  ✓ Journal         100 nodes merged in 0.2s
  ✓ Title        27,815 nodes merged in 8.4s


In [37]:
def load_relationships(driver, relationship_pairs: dict, canonical_schema: list,
                       store_inverse: bool = STORE_INVERSE_EDGES, batch_size: int = BATCH_SIZE):
    inverse_lookup = {(e["source"], e["relation"], e["target"]): e["inverse_relation"] for e in canonical_schema}

    with driver.session() as session:
        for (s_type, rel, t_type), pairs in relationship_pairs.items():
            if not pairs:
                continue
            s_label, t_label = LABELS[s_type], LABELS[t_type]
            s_key, t_key = KEY_PROPS[s_type], KEY_PROPS[t_type]
            rows = [{"s": s, "t": t} for s, t in pairs]

            query = f"""
            UNWIND $rows AS row
            MATCH (s:{s_label} {{{s_key}: row.s}})
            MATCH (t:{t_label} {{{t_key}: row.t}})
            MERGE (s)-[:{rel}]->(t)
            """
            t0 = time.time()
            for batch in chunked(rows, batch_size):
                session.run(query, rows=batch)
            print(f"  \u2713 ({s_label})-[:{rel}]->({t_label}): {len(rows):>8,} in {time.time() - t0:.1f}s")

            inverse_rel = inverse_lookup.get((s_type, rel, t_type))
            if store_inverse and inverse_rel:
                inv_name = re.sub(r"\s+", "_", inverse_rel.upper())
                inv_query = f"""
                UNWIND $rows AS row
                MATCH (s:{s_label} {{{s_key}: row.s}})
                MATCH (t:{t_label} {{{t_key}: row.t}})
                MERGE (t)-[:{inv_name}]->(s)
                """
                for batch in chunked(rows, batch_size):
                    session.run(inv_query, rows=batch)
                print(f"    \u21b3 also stored ({t_label})-[:{inv_name}]->({s_label})")


load_relationships(driver, relationship_pairs, canonical_schema)

  ✓ (Author)-[:WRITES]->(Title):   81,028 in 15.8s
  ✓ (Author)-[:CONTRIBUTES_TO]->(Journal):   66,346 in 9.6s
  ✓ (Author)-[:PUBLISHES_WITH]->(Publisher):   61,940 in 7.8s
  ✓ (Author)-[:AFFILIATED_WITH]->(Place):   62,881 in 7.6s
  ✓ (Author)-[:USES]->(Keyword):  236,412 in 38.5s
  ✓ (Author)-[:PUBLISHES_IN]->(Year):   68,332 in 10.0s
  ✓ (Author)-[:LINKED_BY]->(DOI):   80,865 in 11.9s
  ✓ (Author)-[:ACCESSIBLE_VIA]->(URL):   81,023 in 12.5s
  ✓ (Title)-[:PUBLISHED_IN]->(Journal):   27,815 in 4.1s
  ✓ (Title)-[:PUBLISHED_BY]->(Publisher):   27,815 in 3.2s
  ✓ (Title)-[:PUBLISHED_AT]->(Place):   27,815 in 3.1s
  ✓ (Title)-[:DESCRIBED_BY]->(Keyword):   83,359 in 8.8s
  ✓ (Title)-[:APPEARS_IN]->(Year):   27,815 in 2.9s
  ✓ (Title)-[:IDENTIFIED_BY]->(DOI):   27,749 in 3.2s
  ✓ (Title)-[:AVAILABLE_AT]->(URL):   27,813 in 3.2s
  ✓ (Journal)-[:ISSUED_BY]->(Publisher):      100 in 0.1s
  ✓ (Journal)-[:PUBLISHED_AT]->(Place):      100 in 0.2s
  ✓ (Journal)-[:INDEXED_BY]->(Keyword):   54,630 i

In [38]:
def summarize_graph(driver):
    with driver.session() as session:
        print("Nodes by label:")
        for row in session.run("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n ORDER BY n DESC"):
            print(f"  {row['label']:<10} {row['n']:>8,}")

        print("\nRelationships by type:")
        for row in session.run("MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n ORDER BY n DESC"):
            print(f"  {row['rel']:<20} {row['n']:>8,}")


summarize_graph(driver)

Nodes by label:
  Author       52,821
  Keyword      38,589
  Title        27,815
  URL          27,813
  DOI          27,749
  Journal         100
  Place            35
  Publisher        32
  Year              6

Relationships by type:
  USES                  236,412
  DESCRIBED_BY           83,359
  WRITES                 81,028
  ACCESSIBLE_VIA         81,023
  LINKED_BY              80,865
  PUBLISHES_IN           68,332
  CONTRIBUTES_TO         66,346
  AFFILIATED_WITH        62,881
  PUBLISHES_WITH         61,940
  INDEXED_BY             54,630
  RELEVANT_IN            50,298
  ASSOCIATED_WITH        49,504
  CONNECTED_WITH         47,756
  PUBLISHED_AT           27,915
  PUBLISHED_IN           27,815
  PUBLISHED_BY           27,815
  APPEARS_IN             27,815
  PROVIDES               27,813
  ACCESSIBLE_AT          27,813
  AVAILABLE_AT           27,813
  ASSIGNS                27,749
  REPRESENTED_BY         27,749
  IDENTIFIED_BY          27,749
  ACTIVE_IN               

In [39]:
# Example: everything connected to one Title, two hops out
sample_query = """
MATCH (t:Title)
WITH t ORDER BY t.cites DESC LIMIT 1
MATCH (t)-[r]-(n)
RETURN t.title AS title, type(r) AS relationship, labels(n)[0] AS neighbor_type, coalesce(n.name, toString(n.value)) AS neighbor
"""
with driver.session() as session:
    for row in session.run(sample_query):
        print(f"{row['title'][:60]:<60}  -[{row['relationship']}]->  {row['neighbor_type']}: {row['neighbor']}")

Review of deep learning: concepts, CNN architectures, challe  -[WRITES]->  Author: Amjad J. Humaidi
Review of deep learning: concepts, CNN architectures, challe  -[WRITES]->  Author: Ayad Q. Al-Dujaili
Review of deep learning: concepts, CNN architectures, challe  -[WRITES]->  Author: Jinglan Zhang
Review of deep learning: concepts, CNN architectures, challe  -[WRITES]->  Author: José Santamaría
Review of deep learning: concepts, CNN architectures, challe  -[WRITES]->  Author: Laith Alzubaidi
Review of deep learning: concepts, CNN architectures, challe  -[WRITES]->  Author: Laith Farhan
Review of deep learning: concepts, CNN architectures, challe  -[WRITES]->  Author: Mohammed A. Fadhel
Review of deep learning: concepts, CNN architectures, challe  -[WRITES]->  Author: Muthana Al‐Amidie
Review of deep learning: concepts, CNN architectures, challe  -[WRITES]->  Author: Omran Al-Shamma
Review of deep learning: concepts, CNN architectures, challe  -[WRITES]->  Author: Ye Duan
Review of deep

In [40]:
driver.close()